# ðŸŒ¾ Read the Market, Choose the Farm ðŸ“ˆ

> A deterministic Kaggriculture agent that treats the public shop sequence as an early demand signal and commits to the farm plan best suited to that market.

A strong farm is not only a collection of individually profitable actions. Crops, animals, land, labor, storage and market timing have to support the same economic plan. The complication is that the most profitable plan changes with the shops that appear in town.

This notebook uses a small **mixture of complete strategies**. Both strategies share the same opening, so the agent can observe public demand before making an irreversible expansion decision. It then selects one route and follows it consistently for the rest of the game.

## The market signal

The agent records `town.unlocked_shops` during the shared opening and makes one decision at step 168:

- If a `YARN_STORE` is visible, choose the wool-oriented, high-capacity route.
- If the ordered opening is `ICE_CREAM_SHOP` followed by `YARN_STORE`, keep the balanced route instead. Early ice-cream demand directly rewards its milk, strawberry and wheat portfolio.
- If no Yarn Store is visible, keep the balanced route.

```text
observe public shops through the common prefix
                 |
        Yarn Store visible?
          /             \
        no               yes
        |                 |
 balanced route    Ice Cream -> Yarn?
                       /       \
                     yes        no
                      |          |
               balanced route  high-capacity route
```

This is deliberately a public-state rule. It does not inspect the opponent's identity, hidden inventory, seed, reward or future information.

## What each route contributes

The **balanced route** emphasizes the milk/strawberry/wheat economy and avoids paying for capacity that weak demand may not repay. The **high-capacity route** expands farther and shifts the later economy toward sheep and wool when public demand justifies that investment.

The production routes are supported by a common execution layer:

- deterministic worker and field schedules;
- weed recovery and worker-count alignment;
- feed, room and inventory safeguards;
- market-impact-aware sell ordering;
- bounded public-structure market counters; and
- endgame capacity control and liquidation.

The important design choice is **coherence**: the selector chooses between complete farm economies rather than mixing isolated purchases from incompatible plans.

## Validation

All comparisons used identical seeds in both seats. Results were aggregated by seed block so a favorable seat order could not masquerade as an improvement.

| Test | Result |
|---|---:|
| Fresh selector ablation | 8/8 eligible demand blocks improved; median **+24,862** coins |
| Non-triggered demand blocks | 52/52 had zero paired change |
| Protected fallback, 20 fresh seeds | **17 wins, 3 losses** by paired block |
| Open leader panel, 59 replay controls | **51 wins, 2 losses, 6 ties** by paired block |
| Sealed leader holdout, 21 controls | **19 wins, 1 loss, 1 tie** by paired block |
| Three recent strategy controls, 30 seeds each | **26-3-1**, **23-7-0**, and **30-0-0** |

Every reported validation game completed without an execution failure. Replay-based tests are counterfactual controls: they measure route and shared-market compatibility, but they cannot reproduce every reaction of a live adaptive opponent.

## Attribution

This is a derived and fully attributed agent, not a claim of inventing both production routes.

- The balanced controller is based on Boatlee's public [High-Score 10C/4S Market & Storage](https://www.kaggle.com/code/boatlee/v17-r1-rc2-high-score-10c-4s-market-storage) notebook.
- The high-capacity route is a behavioral reconstruction from Kawashigi's public replay, episode `92521336`, seat 0. It is not their hidden source policy.
- The public-demand selector, ordered Ice Cream/Yarn exception, validation protocol, deterministic packaging and deployment safeguard were developed for this notebook.

Please retain this attribution when reusing the agent.

## Reproducibility

The notebook is CPU-only and does not require internet access. The first code cell contains the complete submission payload; the second compiles it, verifies its SHA-256 digest and builds a deterministic `submission.tar.gz` containing only `main.py`. Fixed archive metadata makes repeated builds byte-for-byte reproducible.

## Limitations

The selector makes one early commitment from a deliberately small signal set. It does not perform continuous replanning, identify opponents or predict future shop unlocks. A previously unseen demand ordering can therefore favor a route that is not represented here. The method is best understood as a robust, interpretable market-conditioned policy rather than a universal game solver.

In [1]:
AGENT_B64 = (
    '...'
)

In [2]:
import base64
import gzip
import hashlib
import io
import tarfile
from pathlib import Path

EXPECTED_MAIN_SHA = "d39dba50793d9777c990347443bf0c481c78adaea86055f6f6b0600dcfcd9f2e"
EXPECTED_ARCHIVE_SHA = "a5f0e99ef483408fb524e7ae7c9c2df0c71fd849a30e4fcc54ef50fc166e3ee8"
payload = base64.b64decode(AGENT_B64)
compile(payload, "main.py", "exec")
assert hashlib.sha256(payload).hexdigest() == EXPECTED_MAIN_SHA
Path("main.py").write_bytes(payload)

with Path("submission.tar.gz").open("wb") as raw:
    with gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as compressed:
        with tarfile.open(fileobj=compressed, mode="w") as archive_out:
            info = tarfile.TarInfo("main.py")
            info.size = len(payload)
            info.mode = 0o644
            info.mtime = 0
            info.uid = info.gid = 0
            info.uname = info.gname = ""
            archive_out.addfile(info, io.BytesIO(payload))

archive_bytes = Path("submission.tar.gz").read_bytes()
assert hashlib.sha256(archive_bytes).hexdigest() == EXPECTED_ARCHIVE_SHA
with tarfile.open("submission.tar.gz", "r:gz") as archive_in:
    assert archive_in.getnames() == ["main.py"]
    assert archive_in.extractfile("main.py").read() == payload
print("main.py:", EXPECTED_MAIN_SHA)
print("submission.tar.gz:", EXPECTED_ARCHIVE_SHA)

main.py: d39dba50793d9777c990347443bf0c481c78adaea86055f6f6b0600dcfcd9f2e
submission.tar.gz: a5f0e99ef483408fb524e7ae7c9c2df0c71fd849a30e4fcc54ef50fc166e3ee8
